Logistics Inventory Management and Demand Forecasting
Project Overview

This project applies Python and machine learning techniques to improve inventory management and demand forecasting in logistics.

Objectives
Analyze historical retail transactions.
Clean and prepare datasets.
Explore product demand patterns.
Forecast future demand.
Categorize products using K-Means clustering.
Support better inventory management decisions.

Dataset: UCI Online Retail II Dataset

Technologies: Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn.

In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

# Upload your Online Retail II Excel dataset
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_excel(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()


In [ ]:

df = df.drop_duplicates()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

df["InvoiceNo"] = df["InvoiceNo"].astype(str)

cancelled = df["InvoiceNo"].str.startswith("C")

sales = df[
    (~cancelled) &
    (df["Quantity"] > 0) &
    (df["Price"] > 0)
].dropna(
    subset=["StockCode", "InvoiceDate"]
).copy()

sales["Revenue"] = (
    sales["Quantity"] * sales["Price"]
)

print("Cleaning completed!")
print("Cleaned records:", len(sales))
sales.head()


In [ ]:

top_products = (
    sales.groupby("StockCode")["Quantity"]
    .sum()
    .nlargest(10)
)

plt.figure(figsize=(12, 5))
top_products.plot(kind="bar", color="steelblue")

plt.title("Top 10 Best-Selling Products")
plt.xlabel("Product")
plt.ylabel("Quantity Sold")
plt.tight_layout()
plt.show()

monthly_sales = (
    sales.set_index("InvoiceDate")
    .resample("MS")["Revenue"]
    .sum()
)

monthly_sales.plot(figsize=(12, 5), color="green")

plt.title("Monthly Revenue Trends")
plt.ylabel("Revenue")
plt.show()


In [ ]:

top_product = (
    sales.groupby("StockCode")["Quantity"]
    .sum().idxmax()
)

product_data = sales[
    sales["StockCode"] == top_product
]

daily = (
    product_data.groupby(
        product_data["InvoiceDate"].dt.floor("D")
    )["Quantity"].sum()
    .asfreq("D", fill_value=0)
)

forecast = pd.DataFrame({"demand": daily})

forecast["lag1"] = forecast["demand"].shift(1)
forecast["lag7"] = forecast["demand"].shift(7)

forecast["rolling7"] = (
    forecast["demand"].shift(1).rolling(7).mean()
)

forecast = forecast.dropna()

X = forecast[["lag1", "lag7", "rolling7"]]
y = forecast["demand"]

split = int(len(forecast) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Forecast MAE:",
      mean_absolute_error(y_test, predictions))

plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test, label="Actual")
plt.plot(y_test.index, predictions, label="Predicted")

plt.title("Actual vs Predicted Demand")
plt.legend()
plt.show()


In [ ]:

features = sales.groupby("StockCode").agg(
    total_demand=("Quantity", "sum"),
    average_demand=("Quantity", "mean"),
    revenue=("Revenue", "sum")
).fillna(0)

scaled = StandardScaler().fit_transform(
    np.log1p(features)
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

features["Cluster"] = kmeans.fit_predict(scaled)

print("Product clustering completed!")

print(features.groupby("Cluster").mean())

features.head()
